# 01 Check Eelbrain-Main Pipeline Setup

This notebook checks the Eelbrain-main pipeline inputs before estimating any TRFs. It is meant to be run cell-by-cell.


In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_eelbrain_main_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_eelbrain_main_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_eelbrain_main_experiment import (
    BIDS_ROOT,
    BIDS_SEGMENT_DURATION,
    PREDICTOR_ROOT,
    SEGMENT_DURATION,
    WAV_SEGMENT_DURATION,
    alice,
)

print(f'Pipeline directory: {PIPELINE_DIR}')
print(f'BIDS root: {BIDS_ROOT}')
print(f'Predictor root: {PREDICTOR_ROOT}')


INFO    :  *** AliceComprehensionEelbrainMain initialized with root /Users/yanyuwoo/Data/bids on 2026-07-20 20:23:12 ***
INFO    :  Using eelbrain 0.43.0a1, mne 1.11.0.
Pipeline directory: /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction/analysis/trf_pipeline
BIDS root: /Users/yanyuwoo/Data/bids
Predictor root: /Users/yanyuwoo/Data/derivatives/predictors


## Subjects and Segment Durations

`SEGMENT_DURATION` is the duration source used by the TRF pipeline. It now comes from BIDS `events.tsv`. WAV duration is shown only as a QC comparison.

In [2]:
subjects = alice.get_field_values('subject')
print(f'N subjects: {len(subjects)}')
print(subjects[:10], '...', subjects[-5:])

duration_rows = []
for segment in sorted(SEGMENT_DURATION, key=int):
    bids_duration = BIDS_SEGMENT_DURATION[segment]
    wav_duration = WAV_SEGMENT_DURATION.get(segment)
    duration_rows.append({
        'segment': segment,
        'duration_source_used_by_pipeline': 'BIDS events.tsv',
        'duration_sec': bids_duration,
        'wav_duration_sec': wav_duration,
        'bids_minus_wav_sec': bids_duration - wav_duration if wav_duration is not None else None,
    })

duration_table = pd.DataFrame(duration_rows).sort_values('segment', key=lambda s: s.astype(int))
duration_table

N subjects: 49
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10'] ... ['45', '46', '47', '48', '49']


,segment,duration_source_used_by_pipeline,duration_sec,wav_duration_sec,bids_minus_wav_sec
0,1,BIDS events.tsv,57.540612,57.540612,0.0
1,2,BIDS events.tsv,60.845193,60.845193,0.0
2,3,BIDS events.tsv,63.259433,63.259433,0.0
3,4,BIDS events.tsv,69.988571,69.988571,0.0
4,5,BIDS events.tsv,66.272540,66.272540,0.0
5,6,BIDS events.tsv,63.777551,63.777551,0.0
6,7,BIDS events.tsv,62.896848,62.896848,0.0
7,8,BIDS events.tsv,57.310612,57.310612,0.0
8,9,BIDS events.tsv,57.226145,57.226145,0.0
9,10,BIDS events.tsv,61.269660,61.269660,0.0


## Predictor Files

The first formal model uses `gammatone-8` files in BIDS derivatives.

In [3]:
predictor_dir = PREDICTOR_ROOT
rows = []
for segment in range(1, 13):
    path = predictor_dir / f'{segment}~gammatone-8.pickle'
    rows.append({'segment': segment, 'path': str(path), 'exists': path.exists()})

predictor_table = pd.DataFrame(rows)
display(predictor_table)
assert predictor_table['exists'].all(), 'Missing gammatone-8 predictor files'


,segment,path,exists
0,1,/Users/yanyuwoo/Data/derivatives/predictors/1~...,True
1,2,/Users/yanyuwoo/Data/derivatives/predictors/2~...,True
2,3,/Users/yanyuwoo/Data/derivatives/predictors/3~...,True
3,4,/Users/yanyuwoo/Data/derivatives/predictors/4~...,True
4,5,/Users/yanyuwoo/Data/derivatives/predictors/5~...,True
5,6,/Users/yanyuwoo/Data/derivatives/predictors/6~...,True
6,7,/Users/yanyuwoo/Data/derivatives/predictors/7~...,True
7,8,/Users/yanyuwoo/Data/derivatives/predictors/8~...,True
8,9,/Users/yanyuwoo/Data/derivatives/predictors/9~...,True
9,10,/Users/yanyuwoo/Data/derivatives/predictors/10...,True


## Events for One Subject

Eelbrain main reads event timing from BIDS/raw data. The pipeline maps the BIDS `trial_type` marker labels to the clean `segment` variable; `stimulus_id` is printed as a QC cross-check.


In [4]:
subject = '01'
events = alice.load_events(subject=subject, raw='0.5-20', epoch='chapter-1')
print(f'Loaded {events.n_cases} events for subject {subject}')
events.head()


Loaded 12 events for subject 01


#,onset,duration,trial_type,value,sample,stimulus_id,subject,segment
0,3.664,58.541,Stimulus/1,1,1832,1,01,1
1,61.292,61.845,Stimulus/2,5,30646,2,01,2
2,122.19,64.259,Stimulus/3,6,61094,3,01,3
3,185.5,70.989,Stimulus/4,7,92750,4,01,4
4,255.54,67.273,Stimulus/5,8,127772,5,01,5
5,321.87,64.778,Stimulus/6,9,160936,6,01,6
6,385.7,63.897,Stimulus/7,10,192850,7,01,7
7,448.66,58.311,Stimulus/8,11,224330,8,01,8
8,506.02,58.226,Stimulus/9,12,253011,9,01,9
9,563.3,62.27,Stimulus/10,2,281651,10,01,10


In [5]:
print('MNE event values:', list(events['value']))
print('BIDS trial types:', list(events['trial_type']))
print('BIDS stimulus IDs:', list(events['stimulus_id']))
print('Clean segment labels:', list(events['segment']))


MNE event values: [np.int64(1), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(2), np.int64(3), np.int64(4)]
BIDS trial types: ['Stimulus/1', 'Stimulus/2', 'Stimulus/3', 'Stimulus/4', 'Stimulus/5', 'Stimulus/6', 'Stimulus/7', 'Stimulus/8', 'Stimulus/9', 'Stimulus/10', 'Stimulus/11', 'Stimulus/12']
BIDS stimulus IDs: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Clean segment labels: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']


## Channel-Type Check

`AUD` should remain in raw data as a `misc` channel, not as an EEG target.

In [6]:
alice.set(subject='01', raw='0.5-20')
raw = alice.load_raw(preload=False)
channel_types = raw.get_channel_types()
print(f'N channels total: {len(raw.ch_names)}')
print(f'N EEG channels: {channel_types.count("eeg")}')
print(f'N MISC channels: {channel_types.count("misc")}')
print(f'AUD present: {"AUD" in raw.ch_names}')
if 'AUD' in raw.ch_names:
    print(f'AUD type: {raw.get_channel_types(picks=["AUD"])[0]}')


Reading 0 ... 366524  =      0.000 ...   733.048 secs...
INFO    :  Raw 0.5-20: filtering for /Users/yanyuwoo/Data/bids/sub-01/eeg/sub-01_task-alice_eeg.vhdr...
N channels total: 62
N EEG channels: 61
N MISC channels: 1
AUD present: True
AUD type: misc
